In [ ]:
# automatically reload modules when scripts/code changes
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import joblib
warnings.filterwarnings('ignore')

sys.path.append('../../scripts')
from data_pipeline import load_and_prepare_data, load_single_channel, test_train_split
from feature_engineering import build_state_vector
import plots, xgb_scripts, evaluate

array(['mode', 'ox/red', 'error', 'control changes', 'Ns changes',
       'counter inc.', 'Ns', 'I Range', 'time/s', 'control/V/mA',
       'Ecell/V', 'I/mA', 'dq/mA.h', '(Q-Qo)/mA.h', '|Energy|/W.h',
       'freq/Hz', '|Z|/Ohm', 'Phase(Z)/deg', 'Q charge/discharge/mA.h',
       'half cycle', 'Energy charge/W.h', 'Energy discharge/W.h',
       'Capacitance charge/µF', 'Capacitance discharge/µF', 'step time/s',
       'z cycle', 'Re(Z)/Ohm', 'Im(Z)/Ohm', 'Re(Y)/Ohm-1', 'Im(Y)/Ohm-1',
       '|Y|/Ohm-1', 'Phase(Y)/deg', 'x', 'Q discharge/mA.h',
       'Q charge/mA.h', 'Capacity/mA.h', 'Efficiency/%', 'control/V',
       'control/mA', 'cycle number', 'P/W', 'R/Ohm'], dtype=object)

In [3]:
def build_action_spectrum(
    df: pd.DataFrame,
    rated_capacity_mAh: float = 4000.0,  # 4 Ah from your header
    ns_charge_cc: int = 3,
    ns_charge_cv: int = 4,
    ns_discharge: int = 8,
    current_pref: str = "control/mA",    # falls back to 'I/mA'
    use_c_rate: bool = True,             # True -> convert to C-rate
    n_cc: int = 64,                      # resampled points from CC charge
    n_cv: int = 64,                      # resampled points from CV charge
    n_dis: int = 64,                     # resampled points from discharge
    use_abs: bool = True,                # take |I| to avoid sign convention issues
):
    """
    Build a fixed-length 'action spectrum' per cycle by resampling the current trace
    during CC charge (Ns=ns_charge_cc), CV charge (Ns=ns_charge_cv), and discharge (Ns=ns_discharge).

    Returns
    -------
    cycles : np.ndarray shape (N,)
    X      : np.ndarray shape (N, n_cc + n_cv + n_dis)
    feature_names : list[str]  length n_cc + n_cv + n_dis

    Notes
    -----
    • Safe timing: use these traces from cycle n to predict Q_{n+1}. For Q_n, use protocol-only features.
    • If a segment is missing in a given cycle, it's filled with NaNs.
    """

    if "cycle number" not in df.columns or "Ns" not in df.columns:
        raise ValueError("Expected columns 'cycle number' and 'Ns' are missing.")

    cur_col = current_pref if current_pref in df.columns else ("I/mA" if "I/mA" in df.columns else None)
    if cur_col is None:
        raise ValueError("Neither 'control/mA' nor 'I/mA' found in dataframe.")

    # Optional time column for time-aware resampling; fall back to index if missing
    t_col = "step time/s" if "step time/s" in df.columns else None

    def _resample_segment(seg: pd.DataFrame, N: int) -> np.ndarray:
        """Resample current segment to N points using step time if available, else index."""
        if seg is None or seg.empty or N <= 0:
            return np.full(N, np.nan, dtype=float)

        y = seg[cur_col].astype(float).values
        if use_abs:
            y = np.abs(y)

        # Convert to C-rate if requested
        if use_c_rate and rated_capacity_mAh and rated_capacity_mAh > 0:
            y = y / rated_capacity_mAh

        # Build the x-axis
        if t_col is not None and t_col in seg.columns:
            t = seg[t_col].astype(float).values
            # guard against degenerate times
            t = t - np.nanmin(t)
            span = np.nanmax(t)
            if not np.isfinite(span) or span <= 0:
                # fall back to index
                x = np.linspace(0.0, 1.0, num=len(y), endpoint=True)
            else:
                x = t / span
        else:
            x = np.linspace(0.0, 1.0, num=len(y), endpoint=True)

        # Resample to a fixed grid
        xi = np.linspace(0.0, 1.0, num=N, endpoint=True)
        # handle all-NaN or length-1 edge cases
        if len(y) == 1 or np.all(~np.isfinite(x)):
            return np.full(N, np.nan, dtype=float)
        # Interpolate only over finite portions
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 2:
            return np.full(N, np.nan, dtype=float)

        try:
            yi = np.interp(xi, x[mask], y[mask])
        except Exception:
            yi = np.full(N, np.nan, dtype=float)
        return yi

    cycles = np.sort(df["cycle number"].dropna().unique())
    rows = []

    for c in cycles:
        d = df[df["cycle number"] == c]
        ch_cc = d[d["Ns"] == ns_charge_cc]
        ch_cv = d[d["Ns"] == ns_charge_cv]
        dis   = d[d["Ns"] == ns_discharge]

        v_cc  = _resample_segment(ch_cc, n_cc)
        v_cv  = _resample_segment(ch_cv, n_cv)
        v_dis = _resample_segment(dis,   n_dis)

        rows.append(np.concatenate([v_cc, v_cv, v_dis], axis=0))

    X = np.vstack(rows)

    unit = "C" if use_c_rate else "mA"
    feature_names = (
        [f"CC_charge_I_{i}_{unit}" for i in range(n_cc)] +
        [f"CV_charge_I_{i}_{unit}" for i in range(n_cv)] +
        [f"Discharge_I_{i}_{unit}" for i in range(n_dis)]
    )

    # optional: replace inf with nan
    X[~np.isfinite(X)] = np.nan
    return cycles, X, feature_names


In [4]:
df = load_single_channel(
    data_folder="../../data/04-03-24",
    channel="A1",
)
df

,mode,ox/red,error,control changes,Ns changes,counter inc.,Ns,I Range,time/s,control/V/mA,...,Q discharge/mA.h,Q charge/mA.h,Capacity/mA.h,Efficiency/%,control/V,control/mA,cycle number,P/W,R/Ohm,channel
340,1.0,0.0,0.0,0.0,0.0,1.0,3.0,112.0,11100.0,8000.00,...,0.0,0.00444,0.00444,0.0,0.00,8000.0,1.0,22.900,0.358,A1
341,1.0,1.0,0.0,0.0,1.0,1.0,3.0,112.0,11100.0,8000.00,...,0.0,0.00888,0.00888,0.0,0.00,8000.0,1.0,22.900,0.359,A1
342,1.0,1.0,0.0,0.0,0.0,1.0,3.0,112.0,11200.0,8000.00,...,0.0,133.00000,133.00000,0.0,0.00,8000.0,1.0,27.500,0.430,A1
343,1.0,1.0,0.0,0.0,0.0,1.0,3.0,112.0,11200.0,8000.00,...,0.0,200.00000,200.00000,0.0,0.00,8000.0,1.0,28.200,0.440,A1
344,1.0,1.0,0.0,0.0,0.0,1.0,3.0,112.0,11300.0,8000.00,...,0.0,267.00000,267.00000,0.0,0.00,8000.0,1.0,28.500,0.446,A1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80444,2.0,0.0,0.0,1.0,0.0,0.0,1.0,113.0,2420000.0,3.49,...,2230.0,0.00000,2230.00000,0.0,3.49,0.0,267.0,-1.110,10.900,A1
80445,2.0,0.0,0.0,1.0,0.0,0.0,1.0,113.0,2420000.0,3.49,...,2230.0,0.00000,2230.00000,0.0,3.49,0.0,267.0,-0.434,27.900,A1
80446,2.0,1.0,0.0,1.0,0.0,0.0,1.0,113.0,2420000.0,3.49,...,2240.0,0.00000,2240.00000,0.0,3.49,0.0,267.0,0.468,26.100,A1
80447,2.0,1.0,0.0,1.0,0.0,0.0,1.0,113.0,2420000.0,3.49,...,2240.0,0.00000,2240.00000,0.0,3.49,0.0,267.0,0.336,36.300,A1


In [5]:
cycles, X, feature_names = build_action_spectrum(df)

In [6]:
X

array([[2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75],
       [2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75],
       [2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75],
       ...,
       [2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75],
       [2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75],
       [2.  , 2.  , 2.  , ..., 3.75, 3.75, 3.75]])